In [1]:
#!/usr/bin/env Rscript
#install.packages("anndata")

In [3]:
requireNamespace("anndata", quietly=TRUE)
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(purrr)
  library(tibble)
  library(edgeR)
  library(limma)
  library(Matrix)
})

source("subsampling.R")

Warning message:
“package ‘dplyr’ was built under R version 4.2.3”
Warning message:
“package ‘tidyr’ was built under R version 4.2.3”
Warning message:
“package ‘purrr’ was built under R version 4.2.3”
Warning message:
“package ‘tibble’ was built under R version 4.2.3”
Warning message:
“package ‘edgeR’ was built under R version 4.2.3”
Warning message:
“package ‘limma’ was built under R version 4.2.3”
Warning message:
“package ‘Matrix’ was built under R version 4.2.3”


In [4]:
filter_cells <- function(adata,
                         min_cells) {
  n_obs_prev <- adata$n_obs
  message("Filter samples with the low number of cells")
  adata <- adata[adata$obs$psbulk_cells >= min_cells]
  message("    n_obs: ", n_obs_prev, "--> ", adata$n_obs)
  return(adata)
}

In [5]:
## VIASH START
par <- list(
  input      = "../../data/sciplex/pseudobulk_processed/srivatsan20_sciplex3_sep_rep.h5ad",
  output_dir = "../../deg_data",   # make sure this exists
  # Testing parameters - set to NULL to run full analysis
  subsampling  = TRUE,            # Enable subsampling mode (test mode)
  max_cell_types = 3,           # Process only first N cell lines
  max_perturbations = 3,        # Process only first N perturbations per cell line
  max_genes = 1000,             # Use only top N variable genes
  specific_times = c(24),       # Analyze only specific time points (NULL for all)
  specific_perturbagens = NULL, # Analyze only these perturbagens (e.g., c("Drug1", "Drug2"))
  min_cells = 0,               # Filter samples with the low number of cells
  design_param = 'separate_replicates' # One of two options to create a design matrix: ('group_all_replicates', 'separate_replicates') 
)

# Example subsampling configurations:
# 1. Quick test (current settings) - ~1-2 minutes
# 2. Single drug test:
#    specific_perturbagens = c("Belinostat"), max_cell_types = 1
# 3. Time course test:
#    specific_times = c(8, 24, 72), max_perturbations = 2
# 4. Full analysis:
#    subsampling = FALSE (or set all limits to NULL)

## VIASH END

In [6]:
# helper to sanitize names for model.matrix/contrasts
clean <- function(x) gsub("[^[:alnum:]_]", "_", x)

# Define which DE results to store in layers
res_cols <- c("logFC", 
              "stdev.unscaled", 
              "stdev.scaled", 
              "CI.L", 
              "CI.R", 
              "AveExpr", 
              "t", 
              "P.Value", 
              "adj.P.Value.within_one_contrast", 
              "adj.P.Value.across_all_contrasts", 
              "B"
             )

# load the full pseudobulk AnnData
adata <- anndata::read_h5ad(par$input)

In [7]:
adata <- filter_cells(adata, par$min_cells)
adata <- subsampling(adata, par)

Filter samples with the low number of cells

    n_obs: 4974--> 4974


⚡ TEST MODE ENABLED ⚡

Original dataset dimensions: 4974 samples x 25 genes

Filtering to time points: 24 hours

Subsetting to 3 perturbations (+ control): DMSO, tazemetostat, JNJ-7706621, ofloxacin

Calculating gene variance for subsetting...

Subset to top 1000 most variable genes

Test mode dataset: 168 samples x 1000 genes


Conditions to analyze:



# A tibble: 12 × 4
   cell_type perturbagen  pert_time_h n_samples
   <fct>     <fct>              <dbl>     <int>
 1 CVCL_0004 DMSO                  24        32
 2 CVCL_0004 JNJ-7706621           24         8
 3 CVCL_0004 ofloxacin             24         8
 4 CVCL_0004 tazemetostat          24         8
 5 CVCL_0023 DMSO                  24        32
 6 CVCL_0023 JNJ-7706621           24         8
 7 CVCL_0023 ofloxacin             24         8
 8 CVCL_0023 tazemetostat          24         8
 9 CVCL_0031 DMSO                  24        32
10 CVCL_0031 JNJ-7706621           24         8
11 CVCL_0031 ofloxacin             24         8
12 CVCL_0031 tazemetostat          24         8


In [8]:
# Start timer
start_time <- Sys.time()

In [9]:
check_for_controls <- function(obs, 
                               col) {
  # Check for required controls
  control <- obs %>%
    filter(is_control==TRUE) %>%
    pull(col) %>%
    unique()

  treated <- obs %>%
    filter(is_control!=TRUE) %>%
    pull(col) %>%
    unique()
    
  missing_controls <- setdiff(treated, control)
    if (length(missing_controls) > 0) {
      warning("Missing controls for '", col, "' columns: ", paste(missing_controls, collapse=", "))
    }
}

In [10]:
filter_controls <- function(ad,
                            col) {
  # Filter controls
  message("Filter controls by ", col, " column")
  n_obs_prev <- ad$n_obs
    
  control <- ad$obs %>%
    filter(is_control==TRUE) %>%
    pull(col) %>%
    unique()

  treated <- ad$obs %>%
    filter(is_control!=TRUE) %>%
    pull(col) %>%
    unique()
  
  redundant_controls <- setdiff(control, treated)
  if (length(redundant_controls) > 0) {
    ad <- ad[!(ad$obs[, col] %in% redundant_controls)]
  }
  
  message("    n_obs: ", n_obs_prev, "--> ",ad$n_obs)
  return(ad)
}

In [11]:
find_controls <- function(control_obs,
                          time,
                          design_param,
                          plate=NULL) {
    
    if (design_param == "group_all_replicates") {
        select_ctrs <- (control_obs$pert_time_h == time)
        if (!any(select_ctrs)) {
            warning("No control found for time ", pert_time_h, "h, skipping ", raw_cond)
            return(NULL)
            }
        
        }
    else if (design_param == "separate_replicates") {
        select_ctrs <- (control_obs$pert_time_h == time) & (control_obs$plate == plate)
        if (!any(select_ctrs)) {
            warning("No control found for time ", pert_time_h, "h, and plate ", plate, ", skipping ", raw_cond)
            return(NULL)
            }
        }
    
    raw_controls <- unique(control_obs[select_ctrs, ]$cond)
    return(raw_controls)
    
}

In [12]:
derive_top_table <- function(fit,
                             raw_control,
                             raw_cond,
                             perturbagen,
                             pert_dose_uM,
                             pert_time_h) {
    contrast <- paste0("cond", clean(raw_cond), " - cond", clean(raw_control))
    tryCatch({
        ctr <- makeContrasts(contrasts = contrast, levels = colnames(coef(fit)))
        
        fit2 <- contrasts.fit(fit, ctr) %>% 
          eBayes(robust = !isTRUE(par$subsampling))  # Skip robust for speed in subsampling mode
        
        top_table <- topTable(fit2, number = Inf, sort = "none", adjust.method="BH", confint=TRUE) %>%
          rownames_to_column("gene") %>%
          mutate(
            control = clean(raw_control),
            cond = clean(raw_cond),
            perturbagen = perturbagen,
            pert_dose_uM = pert_dose_uM,
            pert_time_h = pert_time_h
          )

        
        stdev_unscaled <- data.frame(fit2$stdev.unscaled, check.names = FALSE) %>%
            rownames_to_column("gene") %>%
            rename(stdev.unscaled = !!sym(contrast))

        stdev_scaled <- data.frame(fit2$stdev.unscaled * sqrt(fit2$s2.post), check.names = FALSE) %>%
            rownames_to_column("gene") %>%
            rename(stdev.scaled = !!sym(contrast))

        top_table <- left_join(top_table, stdev_unscaled, by = "gene")
        top_table <- left_join(top_table, stdev_scaled, by = "gene")
        
        return(top_table)
        
      }, error = function(e) {
        warning("Error in contrast for ", raw_cond, ": ", e$message)
        return(NULL)
      })
    }

In [13]:
run_contrasts <- function(fit,
                          control_obs,
                          treated_obs,
                          design_param) {
    # run one contrast per treated cond vs. the matching control@same time
    n_contrasts <- nrow(treated_obs)
    contrast_num <- 0
    
    de_res <- pmap_dfr(treated_obs, function(cond, perturbagen, pert_dose_uM, pert_time_h, plate, raw_cond, ...) {
      contrast_num <<- contrast_num + 1
      if (contrast_num %% 5 == 1 || contrast_num == n_contrasts) {
        message(sprintf("    Running contrast %d/%d", contrast_num, n_contrasts))
      }

      raw_controls <- find_controls(control_obs,
                    pert_time_h,
                    design_param,
                    plate)
    
      # Check if control exists for this time point
      if (is.null(raw_controls)) {
        return(NULL)
      }
      top_tables <- list()
        
      for (raw_control in raw_controls) {
          l <- length(top_tables)
          top_tables[[l + 1]] <- derive_top_table(
                                  fit,
                                  raw_control,
                                  raw_cond,
                                  perturbagen,
                                  pert_dose_uM,
                                  pert_time_h)
        }
      return(bind_rows(top_tables))
    })
    
    
    return(de_res)
    
}

In [14]:
run_dge <- function(ad,
                    obs,
                    design_param) {
  # build DGEList + design
  counts <- Matrix::t(ad$X)
  dge    <- DGEList(counts=counts)
  if (design_param == "group_all_replicates") {
    design <- model.matrix(
      ~ 0 + cond + plate,
      data = obs
      )
    }
  else if (design_param == "separate_replicates") {
    design <- model.matrix(
      ~ 0 + cond,
      data = obs
      )
    }
  # filter genes and normalize
  keep <- filterByExpr(dge, design)
  dge  <- dge[keep, , keep.lib.sizes=FALSE] %>% calcNormFactors()
  
  # voom + lmFit
  v   <- voom(dge, design, plot=FALSE)
  fit <- lmFit(v, design)
  return(fit)
}

In [15]:
get_obs <- function(de_df,
                    treated_obs) {
  # FIX 1: Ensure dose_value and time are preserved as actual values, not factors
  de_unique <- de_df %>% distinct(control, cond)
  obs_out <- treated_obs %>%
    rownames_to_column("id") %>%
    right_join(de_unique, by = "cond") %>%                     # only contrasts with results; keep same order as de_df
    mutate(
      contrast=factor(paste0(cond,' - ', control)),
      pert_dose_uM = as.numeric(as.character(pert_dose_uM)),       # ensure numeric
      pert_time_h       = as.numeric(as.character(pert_time_h))
    ) %>%
    remove_rownames() %>%
    column_to_rownames("contrast") %>%
    select(-control) %>%
    select(-raw_cond) %>%
    select(-cond)
  return(obs_out)
}

In [16]:
get_var <- function(de_df,
                    ad) {
  genes   <- unique(de_df$gene)
  # FIX 2: Carry over all columns from the original var dataframe
  # Get the original var data for these genes
  original_var <- ad$var
  var_out <- original_var[genes, , drop = FALSE]
  
  # If for some reason the genes aren't in the original var, create a basic var
  if (nrow(var_out) == 0 || !all(genes %in% rownames(var_out))) {
    message("  Warning: Some genes not found in original var, creating basic var dataframe")
    var_out <- data.frame(gene = genes, row.names = genes)
    # Try to merge with available var data
    if (nrow(original_var) > 0) {
      available_genes <- intersect(genes, rownames(original_var))
      if (length(available_genes) > 0) {
        var_out[available_genes, ] <- original_var[available_genes, ]
      }
    }
  }
  return(var_out)
}

In [17]:
get_layers <- function(de_df) {
  # Create layers for each DE statistic
  layers <- map(res_cols, function(m) {
    if (!m %in% names(de_df)) {
      warning("Column ", m, " not found in DE results")
      return(NULL)
    }
    
    de_df %>%
      select(gene, cond, !!sym(m)) %>%
      pivot_wider(names_from = gene, values_from = !!sym(m)) %>%
      arrange(match(cond, rownames(obs_out))) %>%
      select(-cond) %>%
      as.matrix()
  }) %>% 
    set_names(res_cols) %>%
    compact()  # Remove NULL entries
  return(layers)
}

In [18]:
cell_types_to_process <- unique(adata$obs$cell_type)
n_cell_types <- length(cell_types_to_process)
cl_num <- 0

for (cl in cell_types_to_process) {
  cl_num <- cl_num + 1
  cl_start <- Sys.time()
  message("\n▶︎ Processing cell_type ", cl_num, "/", n_cell_types, ": ", cl)
  
  # subset to this cell_type
  ad  <- adata[adata$obs$cell_type == cl, ]
  ad <- filter_controls(ad, "pert_time_h")
  if (par$design_param == "separate_replicates") {
      ad <- filter_controls(ad, "plate")
  }
    
  obs <- ad$obs %>%
    mutate(
    raw_cond = perturbation_label,
    cond = factor(clean(perturbation_label)),  # sanitize values here!
    )
  
  # Check for required controls
  check_for_controls(obs, "pert_time_h")
  if (par$design_param == "separate_replicates") {
      check_for_controls(obs, "plate")
  }
  
  cond <- obs$cond
  fit <- run_dge(ad,
          obs,
          par$design_param)
  
    
  # separate the control and treated conds (we won't DE on control-vs-control)
  control_obs <- obs %>%
      distinct(cond, .keep_all = TRUE) %>%
      filter(is_control==TRUE)
    
  treated_obs <- obs %>%
      distinct(cond, .keep_all = TRUE) %>%
      filter(is_control!=TRUE)

  de_res <- run_contrasts(fit,
                        control_obs,
                        treated_obs,
                        par$design_param)
    
  # Skip if no valid DE results
  if (is.null(de_res) || nrow(de_res) == 0) {
  warning("No valid DE results for cell line ", cl)
  next
  }
  
  # adjust p-values globally across all contrasts
  de_df <- de_res %>%
    mutate(
      adj.P.Value.across_all_contrasts = p.adjust(P.Value, method="BH")
    )  %>%
    rename(adj.P.Value.within_one_contrast = adj.P.Val)
  
  obs_out <- get_obs(de_df, treated_obs)
    
  var_out <- get_var(de_df, ad)
  
  layers <- get_layers(de_df)
  
  # carry over global uns if you like
  new_uns <- adata$uns
  
  # assemble and write
  out_adata <- anndata::AnnData(
    obs    = obs_out,
    var    = var_out,
    layers = layers,
    uns    = new_uns
  )
  
  # Create output directory if it doesn't exist
  if (!is.null(par$subsampling) && par$subsampling) {
    output_dir <- file.path(par$output_dir, "subsampling")
  }
  else {
    output_dir <- file.path(par$output_dir, "full")
  }
    
  if (!dir.exists(output_dir)) {
    dir.create(output_dir, recursive = TRUE)
  }
  
  outfile <- file.path(output_dir, paste0(cl, "_de.h5ad"))
      
  message("  Writing: ", outfile)
  out_adata$write_h5ad(outfile, compression = "gzip")
  
  # Show time for this cell line
  cl_time <- difftime(Sys.time(), cl_start, units = "secs")
  message(sprintf("  ✓ Cell line completed in %.1f seconds", cl_time))
}


▶︎ Processing cell_type 1/3: CVCL_0023

Filter controls by pert_time_h column

    n_obs: 56--> 56

Filter controls by plate column

    n_obs: 56--> 32

    Running contrast 1/24

    Running contrast 6/24

    Running contrast 11/24

    Running contrast 16/24

    Running contrast 21/24

    Running contrast 24/24

  Writing: ../../deg_data/subsampling/CVCL_0023_de.h5ad

  ✓ Cell line completed in 1.8 seconds


▶︎ Processing cell_type 2/3: CVCL_0031

Filter controls by pert_time_h column

    n_obs: 56--> 56

Filter controls by plate column

    n_obs: 56--> 32

    Running contrast 1/24

    Running contrast 6/24

    Running contrast 11/24

    Running contrast 16/24

    Running contrast 21/24

    Running contrast 24/24

  Writing: ../../deg_data/subsampling/CVCL_0031_de.h5ad

  ✓ Cell line completed in 1.3 seconds


▶︎ Processing cell_type 3/3: CVCL_0004

Filter controls by pert_time_h column

    n_obs: 56--> 56

Filter controls by plate column

    n_obs: 56--> 32

    Runni

In [19]:
message("DE analysis complete!")

# Show runtime
end_time <- Sys.time()
runtime <- difftime(end_time, start_time, units = "secs")
message(sprintf("\nTotal runtime: %.1f seconds", runtime))

if (!is.null(par$subsampling) && par$subsampling) {
  message("\n📝 Note: This was a TEST RUN with reduced data.")
  message("Set test_mode = FALSE for full analysis.")
}

DE analysis complete!


Total runtime: 48.5 seconds


📝 Note: This was a TEST RUN with reduced data.

Set test_mode = FALSE for full analysis.

